In [2]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.widgets import Button, TextBox
import os, pathlib

%matplotlib qt

# Modify the directory to the project directory
proj_dir = "debug2"
frame_dir = os.path.join(proj_dir, "frame_dump")

# Get list of available frames
frame_files = [f for f in os.listdir(frame_dir) if f.startswith("frame_") and f.endswith(".npz")]
frame_numbers = sorted([int(f.split("_")[1].split(".")[0]) for f in frame_files])

if not frame_numbers:
    raise ValueError("No frame files found in the specified directory")

# Initial parameters
current_frame_idx = 0
current_frame = frame_numbers[current_frame_idx]
clim_min, clim_max = 0, 100
cmap = "viridis"

# Create figure with proper spacing
fig = plt.figure(figsize=(20, 10))
plt.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.2)

# Create main image subplots
ax1 = plt.subplot2grid((3, 4), (0, 0), rowspan=2)
ax2 = plt.subplot2grid((3, 4), (0, 1), rowspan=2, sharex=ax1, sharey=ax1)
ax3 = plt.subplot2grid((3, 4), (0, 2), rowspan=2, sharex=ax1, sharey=ax1)
ax4 = plt.subplot2grid((3, 4), (0, 3), rowspan=2, sharex=ax1, sharey=ax1)

# Function to load frame data
def load_frame(frame_i):
    frame_file = os.path.join(frame_dir, f"frame_{frame_i}.npz")
    if not os.path.exists(frame_file):
        print(f"Warning: Frame {frame_i} not found")
        return None, None, None
    
    with np.load(frame_file) as data:
        HEDM = data["HEDM"]
        Powder = data["Powder"] 
        Background = data["Background"]
    
    return HEDM, Powder, Background

# Global variables to store image objects
im1, im2, im3, im4 = None, None, None, None
cbar = None

# Function to update plots
def update_plots():
    global current_frame, clim_min, clim_max, im1, im2, im3, im4, cbar
    
    HEDM, Powder, Background = load_frame(current_frame)
    
    if HEDM is not None:
        clim = (clim_min, clim_max)
        
        # Update or create images
        if im1 is None:
            im1 = ax1.imshow(HEDM + Powder + Background, cmap=cmap, clim=clim)
            im2 = ax2.imshow(HEDM, cmap=cmap, clim=clim)
            im3 = ax3.imshow(Powder, cmap=cmap, clim=clim)
            im4 = ax4.imshow(Background, cmap=cmap, clim=clim)
            
            # Create colorbar
            cbar = plt.colorbar(im1, ax=[ax1, ax2, ax3, ax4], shrink=0.8, aspect=30)
        else:
            im1.set_array(HEDM + Powder + Background)
            im1.set_clim(clim)
            im2.set_array(HEDM)
            im2.set_clim(clim)
            im3.set_array(Powder)
            im3.set_clim(clim)
            im4.set_array(Background)
            im4.set_clim(clim)
        
        # Set titles
        ax1.set_title(f"Combined - Frame {current_frame}", fontsize=12, fontweight='bold')
        ax2.set_title(f"HEDM - Frame {current_frame}", fontsize=12, fontweight='bold')
        ax3.set_title(f"Powder - Frame {current_frame}", fontsize=12, fontweight='bold')
        ax4.set_title(f"Background - Frame {current_frame}", fontsize=12, fontweight='bold')
        
        # Update frame counter
        frame_counter.set_text(f"Frame: {current_frame} ({current_frame_idx + 1}/{len(frame_numbers)})")
        
        # Redraw
        fig.canvas.draw_idle()

# Create control area
control_y = 0.08
button_height = 0.04
button_width = 0.08

# Frame navigation buttons
ax_first = plt.axes([0.1, control_y, button_width, button_height])
ax_prev = plt.axes([0.2, control_y, button_width, button_height])
ax_next = plt.axes([0.3, control_y, button_width, button_height])
ax_last = plt.axes([0.4, control_y, button_width, button_height])

btn_first = Button(ax_first, 'First', color='lightblue', hovercolor='skyblue')
btn_prev = Button(ax_prev, 'Previous', color='lightblue', hovercolor='skyblue')
btn_next = Button(ax_next, 'Next', color='lightblue', hovercolor='skyblue')
btn_last = Button(ax_last, 'Last', color='lightblue', hovercolor='skyblue')

# Frame counter text
frame_counter = fig.text(0.52, control_y + 0.02, f"Frame: {current_frame} (1/{len(frame_numbers)})", 
                        fontsize=12, ha='center', fontweight='bold')

# Clim controls
text_y = control_y + 0.01
text_width = 0.05
text_height = 0.025

# Clim min
fig.text(0.62, control_y + 0.025, 'Min:', fontsize=10, ha='center')
ax_clim_min = plt.axes([0.64, text_y, text_width, text_height])
textbox_clim_min = TextBox(ax_clim_min, '', initial=str(clim_min))

# Clim max
fig.text(0.71, control_y + 0.025, 'Max:', fontsize=10, ha='center')
ax_clim_max = plt.axes([0.73, text_y, text_width, text_height])
textbox_clim_max = TextBox(ax_clim_max, '', initial=str(clim_max))

# Auto scale and reset buttons
ax_auto = plt.axes([0.8, control_y, button_width, button_height])
ax_reset = plt.axes([0.9, control_y, button_width, button_height])

btn_auto = Button(ax_auto, 'Auto Scale', color='lightgreen', hovercolor='lightcyan')
btn_reset = Button(ax_reset, 'Reset', color='lightyellow', hovercolor='lightcyan')

# Button callback functions
def go_first(event):
    global current_frame_idx, current_frame
    current_frame_idx = 0
    current_frame = frame_numbers[current_frame_idx]
    update_plots()

def go_prev(event):
    global current_frame_idx, current_frame
    if current_frame_idx > 0:
        current_frame_idx -= 1
        current_frame = frame_numbers[current_frame_idx]
        update_plots()

def go_next(event):
    global current_frame_idx, current_frame
    if current_frame_idx < len(frame_numbers) - 1:
        current_frame_idx += 1
        current_frame = frame_numbers[current_frame_idx]
        update_plots()

def go_last(event):
    global current_frame_idx, current_frame
    current_frame_idx = len(frame_numbers) - 1
    current_frame = frame_numbers[current_frame_idx]
    update_plots()

def update_clim_min(text):
    global clim_min
    try:
        new_val = float(text)
        clim_min = new_val
        update_plots()
    except ValueError:
        textbox_clim_min.set_val(str(clim_min))
        print(f"Invalid min value: {text}")

def update_clim_max(text):
    global clim_max
    try:
        new_val = float(text)
        clim_max = new_val
        update_plots()
    except ValueError:
        textbox_clim_max.set_val(str(clim_max))
        print(f"Invalid max value: {text}")

def auto_scale(event):
    global clim_min, clim_max
    HEDM, Powder, Background = load_frame(current_frame)
    if HEDM is not None:
        combined = HEDM + Powder + Background
        clim_min = float(np.min(combined))
        clim_max = float(np.max(combined))
        textbox_clim_min.set_val(f"{clim_min:.1f}")
        textbox_clim_max.set_val(f"{clim_max:.1f}")
        update_plots()

def reset_clim(event):
    global clim_min, clim_max
    clim_min, clim_max = 0, 100
    textbox_clim_min.set_val(str(clim_min))
    textbox_clim_max.set_val(str(clim_max))
    update_plots()

# Connect buttons to functions
btn_first.on_clicked(go_first)
btn_prev.on_clicked(go_prev)
btn_next.on_clicked(go_next)
btn_last.on_clicked(go_last)
btn_auto.on_clicked(auto_scale)
btn_reset.on_clicked(reset_clim)

# Connect text boxes
textbox_clim_min.on_submit(update_clim_min)
textbox_clim_max.on_submit(update_clim_max)

# Keyboard shortcuts
def on_key(event):
    if event.key == 'right':
        go_next(None)
    elif event.key == 'left':
        go_prev(None)
    elif event.key == 'home':
        go_first(None)
    elif event.key == 'end':
        go_last(None)
    elif event.key == 'a':
        auto_scale(None)
    elif event.key == 'r':
        reset_clim(None)

fig.canvas.mpl_connect('key_press_event', on_key)

# Add instructions
instructions = """Controls: First/Previous/Next/Last buttons • Arrow keys (left/right) • Home/End keys
Clim: Enter values in Min/Max boxes • Auto Scale button • Reset button • Keyboard: A (auto), R (reset)"""

fig.text(0.5, 0.02, instructions, fontsize=10, ha='center', style='italic')

# Initial plot
update_plots()

plt.show()

print(f"Available frames: {frame_numbers}")
print("Interactive plot loaded! Use buttons or keyboard shortcuts.")

Available frames: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Interactive plot loaded! Use buttons or keyboard shortcuts.


In [11]:
import numpy as np
from scipy import ndimage
import matplotlib.pyplot as plt

def rotate_about_point(image, angle, center):
    """
    Rotate image about a specific point (can be fractional)
    
    Parameters:
    - image: 2D or 3D numpy array
    - angle: rotation angle in degrees (counterclockwise)
    - center: tuple (x, y) - center of rotation (can be fractional)
    """
    # Get image center
    image_center = np.array(image.shape[:2][::-1]) / 2  # (x, y) format
    
    # Calculate offset from image center to desired center
    offset = np.array(center) - image_center
    
    # Rotate about image center first
    rotated = ndimage.rotate(image, angle, reshape=False, mode='constant', cval=0)
    
    # Then translate to account for the offset
    # We need to rotate the offset vector too
    angle_rad = np.radians(angle)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    
    # Rotation matrix for the offset
    rotated_offset = np.array([
        offset[0] * cos_a - offset[1] * sin_a,
        offset[0] * sin_a + offset[1] * cos_a
    ])
    
    # Final translation
    final_offset = offset - rotated_offset
    
    # Apply translation using ndimage.shift
    result = ndimage.shift(rotated, [-final_offset[1], -final_offset[0]], 
                          mode='constant', cval=0)
    
    return result

In [ ]:
frame_i=2
HEDM, Powder, Background = load_frame(frame_i)
print(f"Frame: {frame_i}")
clim=(0,1000)
BC_Ypx, BC_Zpx = 1030.4223172377785, 1041.6771903247234
frame_rotated_90 = rotate_about_point(Powder, 90, (BC_Ypx, BC_Zpx))
plt.imshow(Powder + frame_rotated_90, cmap=cmap, clim=clim)

Frame: 2


In [15]:
plt.imshow(rotate_about_point(Powder, 90, (BC_Ypx, BC_Zpx)), cmap=cmap, clim=clim)